### IBM QAOA Parameter Setting Analysis ###
This notebook produces tables and generates plots to analyze real IBM Qunatum hardware data while running QAOA on ensembles of the MaxCut problem. Various training strategies are utilized to determine the optimal parameters (angles of QAOA) before running them on hardware. The idea is to create a resource-cost estimation for these different parameter setting strategies and suggest best practices.

In [3]:
%load_ext autoreload
%autoreload 2

# Path imports
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

# IBM QAOA package specific
from src.Processing import QAOAHardware, QAOATraining
from src.Processing import set_data_path
from src.Processing import load_problem_instance
from src.Appox_Ratio_Calc import get_minmax, extract_minmax_args, maxcut_approximation_ratio

# General useful Python libraries
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np  
from pathlib import Path

# Add stochastic-benchmark src to path and import related libraries
sys.path.append('../../src')
import stochastic_benchmark as SB
import bootstrap
import interpolate
import stats
from utils_ws import *
from plotting import Plotting

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### Data Analysis for IBM Quantum Hardware Data ####

In [ ]:
# Select graph to explore
graph_type = "heavy_hex"

# Set data path
data_dir = "/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data"
# Set instance path
inst_path = "/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/instances"
# Set minmax path for approximation ratio calculations later on (specific to evaluations pertaining the Maccut problem)
minmax_path = "/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/minmax_cuts"

# Set hardware data path to graph type
hardware_data_path = set_data_path(data_dir, True, False, graph_type)
print(hardware_data_path)

# Set training data path to graph type
training_data_path = set_data_path(data_dir, False, True, graph_type)
print(training_data_path)

# # Load instance path
# instance_path = load_problem_instance("heavy_hex")
# print(instance_path)

# Select hardware instance parameters to explore:
p_list = [5, 10] # eg. QAOA depths for heavy_hex
num_nodes = 144 # eg. Problem size
instance_list = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] # eg. instance index (last index only)

# Load heavy_hex instances for the above parameter combination
instance_paths_hardware = []

for p in p_list:
    for instance in instance_list:
        instance_paths_hardware.extend(QAOAHardware.locate_hardware_instance(hardware_data_path, 
        graph_type, str(instance), str(num_nodes), str(p))) # extend ensures flat list
print(instance_paths_hardware)
print(len(instance_paths_hardware)) 

In [ ]:
all_instances_data_hardware = []

for instance_path in instance_paths_hardware:
    all_instances_data_hardware.extend(QAOAHardware.load_hardware_instance(instance_path))

print(all_instances_data_hardware)

# Expnad each object into data columns
all_data_dicts_hardware = []

for object in all_instances_data_hardware:
    data_dict_hardware = {

        "instance_name" : object.instance_name,
        "QPU_time" : object.QPU_time,
        "num_shots" : object.num_shots,
        "problem_class" : object.problem_class,
        "training_method" : object.training_method,
        "expected_energy" : object.expected_energy,
        "result_file" : object.result_file
    }
    all_data_dicts_hardware.append(data_dict_hardware)

In [ ]:
# Creata a DataFrame to document the extracted hardware data  
df_hardware = pd.DataFrame(all_data_dicts_hardware)
print(df_hardware.head())

In [ ]:
# Compute approximation ratio for each object and append to the DataFrame
minmax_cache = {} # Since the same instance is present in multiple rows of the DataFrame, just load and store it once
approximation_ratio = []

for _, row in df_hardware.iterrows():
    instance_name = row["instance_name"]
    energy = row["expected_energy"] 
    instance = instance_name[:3]
    if instance not in minmax_cache:
        minmax_instance_path = get_minmax(minmax_path, graph_type, instance, num_nodes) # Extract specific minmax_instance path
        minmax_cache[instance] = extract_minmax_args(minmax_instance_paths) # Cache args for the speciic instance
    min_cut, max_cut, sum_weights = minmax_cache[instance] # Extract minmax args for the specific instance
    approximation_ratio.append(maxcut_approximation_ratio(min_cut, max_cut, sum_weights, energy)) # Evlaute approximation ratio for that row
df_hardware["approximation_ratio"] = approximation_ratio

print(df_hardware.head())
    

#### Data Analysis for IBM Quantum QAOA Parameter Training Data ####

In [ ]:
# Set trainer and evaluator parameters
trainer = ["F", "FA", "TQA", "RTS", "I"]
evaluator = ["SV", "MPS", "PP"]
bond_dimension = ["16", "20", "24", "32"]
max_weight = ["4", "6"]


instance_paths_training = []

for p in p_list:
    for instnace in instance_list:
        instance_paths_training.append(QAOATraining.locate_training_instance(training_data_path, graph_type, instance, num_nodes, p))